# batchnorm-running-stats — faded example 3: Confirm train-mode forward mutates the running buffer (complete the update)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `batchnorm-running-stats`. Running the beacon reports progress on the `CNN: BatchNorm running stats` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: BatchNorm running stats` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`batchnorm-running-stats`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "batchnorm-running-stats"
DD_SUBTOPIC = "CNN: BatchNorm running stats"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A `.train()`-mode BatchNorm forward does two things at once: it normalizes by the current batch stats AND mutates `running_mean` / `running_var` in place via the EMA. After the call, the buffers have moved toward the batch statistics. Capturing a snapshot before and after the forward lets you observe this side effect directly.

## Faded exercise 3

Implement `train_forward_then_snapshot(x, bn)`: with `bn` in train mode, snapshot `running_mean` before the forward, call `bn(x)` once, then snapshot `running_mean` after. Return `(before, after, y)`. The before-snapshot and the return are given; you must complete the line that performs the train-mode forward pass (which both normalizes and updates the buffers).

**Fill in:** the train-mode forward call bn(x) that normalizes by batch stats and updates the running buffers in place

In [ ]:
def train_forward_then_snapshot(x, bn):
    bn.train()
    before = bn.running_mean.clone()
    y = bn(x)
    after = bn.running_mean.clone()
    return before, after, y

t.manual_seed(0)
bn = t.nn.BatchNorm2d(3)
x = t.randn(6, 3, 5, 5) + 4.0
before, after, y = train_forward_then_snapshot(x, bn)
print("before:", before.round(decimals=4))
print("after :", after.round(decimals=4))
print("buffer moved:", not t.allclose(before, after))


def _test():
    t.manual_seed(0)
    bn = t.nn.BatchNorm2d(3)
    x = t.randn(6, 3, 5, 5) + 4.0
    before, after, y = train_forward_then_snapshot(x, bn)
    # buffer must have moved away from initial zeros toward the batch mean
    assert not t.allclose(before, after, atol=1e-6), "running_mean did not update in train mode"
    # expected EMA update with default momentum 0.1 from zeros
    batch_mean = x.mean(dim=(0, 2, 3))
    expected_after = (1 - 0.1) * before + 0.1 * batch_mean
    assert t.allclose(after, expected_after, atol=1e-5), "running_mean update does not match EMA"
    assert y.shape == x.shape, "output shape wrong"


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def train_forward_then_snapshot(x, bn):
    bn.train()
    before = bn.running_mean.clone()
    y = bn(x)
    after = bn.running_mean.clone()
    return before, after, y

t.manual_seed(0)
bn = t.nn.BatchNorm2d(3)
x = t.randn(6, 3, 5, 5) + 4.0
before, after, y = train_forward_then_snapshot(x, bn)
print("before:", before.round(decimals=4))
print("after :", after.round(decimals=4))
print("buffer moved:", not t.allclose(before, after))
```
</details>